In [11]:
import pandas as pd

df = pd.read_csv('mismatch_summary.tsv', sep='\t')
df2 = pd.read_csv('matched_summary.tsv', sep='\t')

scan	DB_search_score	precursor_score	DB_proteins	precursor_proteins	precursor_better	charge	peptide	peptide_demod	peptide_length	reason_mismatch	MinNTermAdd	minNTermSubtract	MinCTermAdd	minCTermSubtract
6352	-25.036301	106.825996	Q86Y38	P61224;E7ESV4;F5GWU8;F5GX62;F5GYB5;F5H004;F5H077;F5H0B7;F5H491;F5H500;F5H6R7;F5H7Y6	True	2	KQVEVDAQQC+57.021MLEILDTAGTEQ	KQVEVDAQQCMLEILDTAGTEQ	22	non_standard_modification	0	0	5	21
6353	-23.2418	103.260002	Q9H0U3-2;Q9H0U3;A0A087WU53;A0A8I5KUC4;A0A8I5KY62;A0A8I5KYH1;A0A8I5QJJ8;A0A8I5QJM4;A0A8I5QKX7	P61224;E7ESV4;F5GWU8;F5GX62;F5GYB5;F5H004;F5H077;F5H0B7;F5H491;F5H500;F5H6R7;F5H7Y6	True	2	KQVEVDAQQC+57.021MLEILDTAGTEQ	KQVEVDAQQCMLEILDTAGTEQ	22	non_standard_modification	0	0	5	21

In [12]:
# df_filtered = df[(df['precursor_better'] == True) & (df['precursor_proteins'].str.contains(';') == False)]

In [13]:
# df_filtered.head()

In [14]:
prec_protein = set(df['precursor_proteins'].dropna().str.split(r'[;-]').str[0])
# print(f"Number of unique precursor proteins: {len(prec_protein)}")
print(f"Number of unique precursor proteins: {len(prec_protein)}")

db_protein = set(df2['precursor_proteins'].dropna().str.split(r'[;-]').str[0])
print(f"Number of unique DB proteins: {len(db_protein)}")

Number of unique precursor proteins: 331
Number of unique DB proteins: 454


In [15]:
import re

# Create a dictionary to store the data for each precursor protein
#iterate through filtered df each row, make a new df with columns: precursor_protein, precursor_id_number - this is the number of rows(peptides) that has that protein in precursor_proteins, [(evidence_peptides,scan_number) - this is a tuple with multiple peptides and scan ]

protein_data = {}

for idx, row in df.iterrows():
    # Create a dictionary to store the data for each precursor protein
    #iterate through filtered df each row, make a new df with columns: precursor_protein, precursor_id_number - this is the number of rows(peptides) that has that protein in precursor_proteins, [(evidence_peptides,scan_number) - this is a tuple with multiple peptides and scan ]

    proteins = row['precursor_proteins']
    if pd.isna(proteins):
        continue
    protein = re.split(r'[;-]', proteins)[0]
    peptide = row['peptide_demod']
    if len(peptide) <9:
        continue
    scan = row['scan']
    reason_mismatch = row['reason_mismatch']
    evidence = False
    source = 'Missed'

    
    if protein not in protein_data:
        protein_data[protein] = {
            'precursor_protein': protein,
            'precursor_id_number': 0,
            'evidence_peptides_scans': []
        }
    
    protein_data[protein]['precursor_id_number'] += 1
    protein_data[protein]['evidence_peptides_scans'].append((peptide, scan,reason_mismatch,evidence,source))

for protein, data in protein_data.items():
    # print(data)
    peptides_temp = []
    peptides_temp_atomic = []
    peptides_atomic_max_l_dict = {}
    for peptide, _, _, _, _ in protein_data[protein]['evidence_peptides_scans']:
        peptides_temp.append(peptide)

    # initialize peptide dict
    for p in peptides_temp:
        peptides_atomic_max_l_dict[p] = len(p)  # Initialize the dictionary with the length of each peptide

    peptides_to_keep = set(peptides_temp)  # Create a set to track peptides to keep
    for p1 in list(peptides_temp):  # Iterate over the peptides in peptides_temp
        if len(p1) < 9:
            peptides_to_keep.discard(p1)  # Mark p1 for removal if its length is less than 9
            continue
        for p2 in list(peptides_temp):  # Compare p1 with other peptides in peptides_temp
            if p1 in p2 and p1 != p2:
                # Check if p1 is contained within p2 and is not the same as p2
                if peptides_atomic_max_l_dict[p1] < len(p2):
                    # print(f"Peptide {p1} is contained in {p2}")
                    peptides_atomic_max_l_dict[p1] = len(p2)

                    if p1 not in peptides_temp_atomic:
                        peptides_temp_atomic.append(p1)
                    peptides_to_keep.discard(p1)  # Mark p1 for removal if it is smaller and overlaps with p2
                continue

    peptides_temp = list(peptides_to_keep)  # Update peptides_temp with the peptides to keep
    peptides_temp = list(set(peptides_temp))
    
    # print(peptides_temp)
 
    data['n_noncontained_le9'] = len(peptides_temp)

    for peptide, _, _, evidence, _ in protein_data[protein]['evidence_peptides_scans']:
        if peptide in peptides_temp:
            for i, (pep, scan, reason, _, _) in enumerate(protein_data[protein]['evidence_peptides_scans']):
                if pep == peptide:
                    protein_data[protein]['evidence_peptides_scans'][i] = (pep, scan, reason, True, 'Missed')
# Create the new dataframe
df_protein_summary = pd.DataFrame(protein_data.values())

In [16]:
df_protein_summary.to_csv('mismatched_protein_level.tsv', sep='\t', index=False)

In [17]:
matched_protein_data = {}

for idx, row in df2.iterrows():
    proteins = row['precursor_proteins']
    if pd.isna(proteins):
        continue
    protein = re.split(r'[;-]', proteins)[0]
    peptide = row['peptide_demod']
    if len(peptide) < 9:
        continue
    scan = row['scan']
    reason_mismatch = row['reason_mismatch']
    evidence = False
    source = 'Matched'

    if protein not in matched_protein_data:
        matched_protein_data[protein] = {
            'precursor_protein': protein,
            'precursor_id_number': 0,
            'evidence_peptides_scans': []
        }

    matched_protein_data[protein]['precursor_id_number'] += 1
    matched_protein_data[protein]['evidence_peptides_scans'].append((peptide, scan, reason_mismatch, evidence, source))

for protein, data in protein_data.items():
    # print(data)
    peptides_temp = []
    peptides_temp_atomic = []
    peptides_atomic_max_l_dict = {}
    for peptide, _, _, _, _ in protein_data[protein]['evidence_peptides_scans']:
        peptides_temp.append(peptide)

    # initialize peptide dict
    for p in peptides_temp:
        peptides_atomic_max_l_dict[p] = len(p)  # Initialize the dictionary with the length of each peptide

    peptides_to_keep = set(peptides_temp)  # Create a set to track peptides to keep
    for p1 in list(peptides_temp):  # Iterate over the peptides in peptides_temp
        if len(p1) < 9:
            peptides_to_keep.discard(p1)  # Mark p1 for removal if its length is less than 9
            continue
        for p2 in list(peptides_temp):  # Compare p1 with other peptides in peptides_temp
            if p1 in p2 and p1 != p2:
                # Check if p1 is contained within p2 and is not the same as p2
                if peptides_atomic_max_l_dict[p1] < len(p2):
                    # print(f"Peptide {p1} is contained in {p2}")
                    peptides_atomic_max_l_dict[p1] = len(p2)

                    if p1 not in peptides_temp_atomic:
                        peptides_temp_atomic.append(p1)
                    peptides_to_keep.discard(p1)  # Mark p1 for removal if it is smaller and overlaps with p2
                continue

    peptides_temp = list(peptides_to_keep)  # Update peptides_temp with the peptides to keep
    peptides_temp = list(set(peptides_temp))
    
    # print(peptides_temp)
 
    data['n_noncontained_le9'] = len(peptides_temp)

    for peptide, _, _, _, _ in data['evidence_peptides_scans']:
        if peptide in peptides_temp:
            for i, (pep, scan, reason, _, _) in enumerate(data['evidence_peptides_scans']):
                if pep == peptide:
                    data['evidence_peptides_scans'][i] = (pep, scan, reason, True, 'Matched')

df_matched_protein_summary = pd.DataFrame(matched_protein_data.values())
df_matched_protein_summary.to_csv('matched_protein_level.tsv', sep='\t', index=False)

In [18]:
mismatch_summary = df_protein_summary.rename(columns={
    'precursor_id_number': 'mismatch_precursor_id_number',
    'evidence_peptides_scans': 'mismatch_evidence_peptides_scans',
    'n_noncontained_le9': 'mismatch_n_noncontained_le9'
})

matched_summary = df_matched_protein_summary.rename(columns={
    'precursor_id_number': 'matched_precursor_id_number',
    'evidence_peptides_scans': 'matched_evidence_peptides_scans',
    'n_noncontained_le9': 'matched_n_noncontained_le9'
})

combined_protein_summary = pd.merge(
    mismatch_summary,
    matched_summary,
    on='precursor_protein',
    how='outer'
)

def _ensure_list(x):
    return x if isinstance(x, list) else []

combined_protein_summary['mismatch_evidence_peptides_scans'] = \
    combined_protein_summary['mismatch_evidence_peptides_scans'].apply(_ensure_list)
combined_protein_summary['matched_evidence_peptides_scans'] = \
    combined_protein_summary['matched_evidence_peptides_scans'].apply(_ensure_list)

combined_protein_summary['evidence_peptides_scans'] = (
    combined_protein_summary['mismatch_evidence_peptides_scans']
    + combined_protein_summary['matched_evidence_peptides_scans']
)

combined_protein_summary.to_csv('combined_protein_level.tsv', sep='\t', index=False)


In [19]:
def _atomic_noncontained(peptide_records):
    peptides_temp = [
        peptide for peptide, *_ in peptide_records
        if isinstance(peptide, str) and len(peptide) >= 9
    ]
    peptides_atomic_max_l_dict = {p: len(p) for p in peptides_temp}  # Initialize the dictionary with the length of each peptide

    peptides_to_keep = set(peptides_temp)  # Create a set to track peptides to keep
    for p1 in list(peptides_temp):  # Iterate over the peptides in peptides_temp
        if len(p1) < 9:
            peptides_to_keep.discard(p1)  # Mark p1 for removal if its length is less than 9
            continue
        for p2 in list(peptides_temp):  # Compare p1 with other peptides in peptides_temp
            if p1 in p2 and p1 != p2:
                # Check if p1 is contained within p2 and is not the same as p2
                if peptides_atomic_max_l_dict[p1] < len(p2):
                    peptides_atomic_max_l_dict[p1] = len(p2)
                    peptides_to_keep.discard(p1)  # Mark p1 for removal if it is smaller and overlaps with p2
                continue

    peptides_temp = list(peptides_to_keep)  # Update peptides_temp with the peptides to keep
    return set(peptides_temp)

for idx, row in combined_protein_summary.iterrows():
    records = row['evidence_peptides_scans']
    atomic_peptides = _atomic_noncontained(records)
    combined_protein_summary.at[idx, 'combined_n_noncontained_le9'] = len(atomic_peptides)

    for i, (pep, scan, reason, _, source) in enumerate(records):
        records[i] = (pep, scan, reason, pep in atomic_peptides, source)

combined_protein_summary['evidence_peptides_scans'] = combined_protein_summary['evidence_peptides_scans']


combined_protein_summary.to_csv('combined_protein_level.tsv', sep='\t', index=False)